In [ ]:
import base64
from io import BytesIO
import json
import sqlite3
import gradio as gr
from openai import OpenAI
from PIL import Image

client = OpenAI()
MODEL = "llama3.2"
DB = "prices.db"

system_message = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)


# 1. Initialize SQLite database
def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
        )
        sample_prices = [
            ("london", 799.0),
            ("paris", 899.0),
            ("tokyo", 1400.0),
            ("berlin", 499.0),
        ]
        cursor.executemany(
            "INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)",
            sample_prices,
        )
        conn.commit()


init_db()


# 2. Database lookup tool
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (destination_city.lower(),),
        )
        result = cursor.fetchone()
        return f"${result[0]}" if result else "No price data available for this city"


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]


# 3. Multimodal Output Helpers: Destination Image (DALL-E 3) & Voice (TTS)


In [ ]:
def artist(city):
    image_response = client.images.generate(
        model="dall-e-3",
        prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_data = base64.b64decode(image_response.data[0].b64_json)
    return Image.open(BytesIO(image_data))


def talker(message):
    response = client.audio.speech.create(
        model="llama3.2",
        voice="onyx",
        input=message,
    )
    # audio_path = "reply.mp3"
    # response.stream_to_file(audio_path)
    return response.text


# 4. Multimodal Input Helpers: Image Encoding & Audio Transcription
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def transcribe_audio(audio_path):
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", file=audio_file
        )
    return transcript.text


# 5. Tool call processor
def handle_tool_calls(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append(
                {
                    "role": "tool",
                    "content": price_details,
                    "tool_call_id": tool_call.id,
                }
            )
    return responses, cities


# 6. Process multimodal input (text + images + audio files) into Chatbot & OpenAI format
def add_user_input(user_msg, chat_history, api_messages):
    text = user_msg.get("text", "")
    files = user_msg.get("files", [])

    content_parts = []
    display_text = text

    for file_path in files:
        lower = file_path.lower()
        if lower.endswith((".png", ".jpg", ".jpeg", ".webp")):
            b64 = encode_image(file_path)
            content_parts.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
                }
            )
            chat_history.append({"role": "user", "content": {"path": file_path}})
        elif lower.endswith((".mp3", ".wav", ".m4a", ".ogg")):
            spoken_text = transcribe_audio(file_path)
            display_text = f"{display_text} {spoken_text}".strip()

    if display_text:
        content_parts.insert(0, {"type": "text", "text": display_text})
        chat_history.append({"role": "user", "content": display_text})

    api_messages.append({"role": "user", "content": content_parts})
    return gr.MultimodalTextbox(value=None), chat_history, api_messages


# 7. Generate assistant reply, voice audio, and destination image


In [ ]:
def bot_reply(chat_history, api_messages):
    messages = [{"role": "system", "content": system_message}] + api_messages
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses, called_cities = handle_tool_calls(assistant_message)
        cities.extend(called_cities)
        messages.append(assistant_message)
        messages.extend(tool_responses)
        messages.append({"role":"user","content":messages})
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    reply = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})
    api_messages.append({"role": "assistant", "content": reply})

    voice = talker(reply)
    if cities:
        image = artist(cities[0])

    return chat_history, api_messages, voice, image



In [ ]:
# from loaddot import ollama  # pyright: ignore[reportMissingImports]
import gradio as gr
from openai import OpenAI
MODEL="llama3.2"
ollama=OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

def put_message_in_chatbot(message,history):
    return "",history+[{"role":"user","content":message}]


In [ ]:
    
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    relevant_system_message=system_message
    messages=[{"role":"system","content":relevant_system_message}]+history+[{"role":"user","content":message}]
    response=client.chat.completions.create( model=MODEL,messages=messages,tools=tools)
    image_path=None
    if response.choices[0].finish_reason=="tool_calls":
     message=response.choices[0].message
     tool_response,city=handle_tool_call(message)
     message.append(message)
     message.append(tool_response)
     image_path=artist(city)
     response=client.chat.completions.create( model=MODEL,messages=messages)
    # stream=ollama.chat.completions.create(model=MODEL,messages=messages,stream=True)
    reply=response.choices[0].message.content
    history.append [{"role":"assistant","content":reply}] 
    voice=talker(reply)
   
    # for chunk in stream:
    #   history[-1]["content"]+=chunk.choices[0].delta.content or ""
    return history,voice,image_path  # pyright: ignore[reportUndefinedVariable]


In [ ]:
# 8. Gradio Blocks Layout
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=100, type="messages",allow_tags=False)
        image_path = gr.Image(height=100, interactive=False)
    with gr.Row():
        audio_outpath = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot,  # pyright: ignore[reportUndefinedVariable]
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        fn=chat,
        inputs=[message,chatbot],
        outputs=[chatbot, audio_outpath, image_path],
    )
ui.launch(inbrowser=True)





In [ ]:
import os
import time
from io import BytesIO
from PIL import Image
import gradio as gr
# from google import genai
# from google.genai import types

# Initialize the Gemini client (uses GEMINI_API_KEY from environment)
client = genai.Client()

OUTPUT_DIR = "chatbot_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def generate_image(prompt: str, file_id: int) -> str | None:
    """Generates an image using Imagen 3 and returns the local file path."""
    try:
        result = client.models.generate_images(
            model="imagen-3.0-generate-002",
            prompt=prompt,
            config="types".GenerateImagesConfig(
                number_of_images=1,
                output_mime_type="image/jpeg",
                aspect_ratio="1:1",
            ),
        )
        image_path = os.path.join(OUTPUT_DIR, f"img_{file_id}.jpg")
        for generated_image in result.generated_images:
            image = Image.open(BytesIO(generated_image.image.image_bytes))
            image.save(image_path)
            return image_path
    except Exception as e:
        print(f"Image generation error: {e}")
    return None


def generate_audio(text: str, file_id: int) -> str | None:
    """Converts text to speech using Gemini native audio and returns the file path."""
    try:
        audio_path = os.path.join(OUTPUT_DIR, f"aud_{file_id}.wav")
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Read the following text aloud naturally: {text}",
            config=types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name="Puck"
                        )
                    )
                ),
            ),
        )
        for part in response.candidates[0].content.parts:
            if part.inline_data:
                with open(audio_path, "wb") as f:
                    f.write(part.inline_data.data)
                return audio_path
    except Exception as e:
        print(f"Audio generation error: {e}")
    return None


# Step 1: Put user message in chatbot and clear the textbox
def put_message_in_chatbot(message: str, history: list[dict]):
    if not message.strip():
        return "", history
    history = history or []
    history.append({"role": "user", "content": message})
    return "", history


# Step 2: Generate response, audio, and image, then return (history, audio_path, image_path)
def chat(message: str, history: list[dict]):
    if not history or history[-1]["role"] != "user":
        return history, None, None

    # Extract the user's message from history (since put_message_in_chatbot cleared `message`)
    user_prompt = history[-1]["content"]
    file_id = int(time.time() * 1000)

    # Convert Gradio 'messages' history into Gemini Content format for multi-turn context
    gemini_contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        gemini_contents.append(
            types.Content(
                role=role,
                parts=[types.Part.from_text(text=msg["content"])],
            )
        )

    # 1. Generate text reply using conversation history
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=gemini_contents,
    )
    reply_text = response.text

    # 2. Generate audio from the assistant's reply
    audio_path = generate_audio(reply_text, file_id)

    # 3. Generate an accompanying image based on the user's prompt
    image_path = generate_image(user_prompt, file_id)

    # 4. Append assistant reply to Gradio history
    history.append({"role": "assistant", "content": reply_text})

    # Return matches outputs=[chatbot, audio_outpath, image_path]
    return history, audio_path, image_path


# 8. Gradio Blocks Layout
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=300, type="messages", allow_tags=False)
        image_path = gr.Image(height=300, interactive=False)
    with gr.Row():
        audio_outpath = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        fn=chat,
        inputs=[message, chatbot],
        outputs=[chatbot, audio_outpath, image_path],
    )

if __name__ == "__main__":
    ui.launch(inbrowser=True)


ModuleNotFoundError: No module named 'google'

In [2]:
import os
import time
import asyncio
from PIL import Image, ImageDraw
import gradio as gr
import ollama  
import edge_tts  

OUTPUT_DIR = "chatbot_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Change this to any model you have pulled in Ollama (e.g., "llama3.2", "mistral", "gemma2")
OLLAMA_MODEL = "llama3.2"

# Optional: Load local Stable Diffusion if installed
try:
    import torch
    from diffusers import AutoPipelineForText2Image  

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    sd_pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sd-turbo", torch_dtype=dtype, variant="fp16" if device == "cuda" else None
    ).to(device)
    HAS_DIFFUSERS = True
except Exception:
    HAS_DIFFUSERS = False


def generate_image(prompt: str, file_id: int) -> str | None:
    """Generates an image locally via Diffusers if available, or creates a summary card."""
    image_path = os.path.join(OUTPUT_DIR, f"img_{file_id}.jpg")
    try:
        if HAS_DIFFUSERS:
            image = sd_pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
            image.save(image_path)
            return image_path
        else:
            # Fallback visual card if local GPU image generation isn't installed
            img = Image.new("RGB", (512, 512), color=(30, 41, 59))
            draw = ImageDraw.Draw(img)
            draw.text((24, 240), f"Prompt: {prompt[:55]}...", fill=(226, 232, 240))
            img.save(image_path)
            return image_path
    except Exception as e:
        print(f"Image generation error: {e}")
    return None


async def _save_tts_audio(text: str, audio_path: str):
    """Uses Microsoft Edge neural TTS to generate natural speech without an API key."""
    communicate = edge_tts.Communicate(text, voice="en-US-AriaNeural")
    await communicate.save(audio_path)


def generate_audio(text: str, file_id: int) -> str | None:
    """Converts text to speech and returns the saved .mp3 file path."""
    try:
        audio_path = os.path.join(OUTPUT_DIR, f"aud_{file_id}.mp3")
        asyncio.run(_save_tts_audio(text, audio_path))
        return audio_path
    except Exception as e:
        print(f"Audio generation error: {e}")
    return None


# Step 1: Put user message in chatbot and clear the textbox
def put_message_in_chatbot(message: str, history: list[dict]):
    if not message.strip():
        return "", history
    history = history or []
    history.append({"role": "user", "content": message})
    return "", history


# Step 2: Generate Ollama response, audio, and image, then return (history, audio_path, image_path)
def chat(message: str, history: list[dict]):
    if not history or history[-1]["role"] != "user":
        return history, None, None

    user_prompt = history[-1]["content"]
    file_id = int(time.time() * 1000)

    # Gradio's `type="messages"` format ([{"role": ..., "content": ...}])
    # matches Ollama's messages format directly!
    ollama_messages = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in history
    ]

    # 1. Generate text reply from local Ollama model
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=ollama_messages,
        )
        reply_text = response["message"]["content"]
    except Exception as e:
        reply_text = f"Ollama error (is `ollama serve` running?): {e}"

    # 2. Generate audio from the assistant's reply
    audio_path = generate_audio(reply_text, file_id)

    # 3. Generate an accompanying image
    image_path = generate_image(user_prompt, file_id)

    # 4. Append assistant reply to Gradio history
    history.append({"role": "assistant", "content": reply_text})

    # Return matches outputs=[chatbot, audio_outpath, image_path]
    return history, audio_path, image_path


# 8. Gradio Blocks Layout
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=300, type="messages", allow_tags=False)
        image_path = gr.Image(height=300, interactive=False)
    with gr.Row():
        audio_outpath = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        fn=chat,
        inputs=[message, chatbot],
        outputs=[chatbot, audio_outpath, image_path],
    )

if __name__ == "__main__":
    ui.launch(inbrowser=True)


C:\Users\DELL\AppData\Local\Temp\ipykernel_6240\2374578180.py:117: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=300, type="messages", allow_tags=False)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [7]:
import ollama
if ollama:
    print(f"ollama found")
else:
        print(f"No ollama")


ollama found


In [5]:
import sys
!{sys.executable} -m pip install ollama edge-tts

  Using cached ollama-0.6.3-py3-none-any.whl.metadata (7.6 kB)
  Using cached edge_tts-7.3.1-py3-none-any.whl.metadata (6.1 kB)
  Using cached aiohttp-3.14.5-cp312-cp312-win_amd64.whl.metadata (8.5 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-7.2.1-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached propcache-0.5.4-cp312-cp312-win_amd64.whl.metadata (25 kB)
  Using cached yarl-1.25.1-cp312-cp312-win_amd64.whl.metadata (115 kB)
Using cached ollama-0.6.3-py3-none-any.whl (16 kB)
Using cached edge_tts-7.3.1-py3-none-any.whl (38 kB)
Using cached aiohttp-3.14.5-cp312-cp312-win_amd64.whl (506 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Using cached ai


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
